# AquaMind AI – Water Demand Forecasting (NEERARAN)
### Production-Grade, Leak-Free Hourly Multi-Horizon Water Consumption Module

This notebook implements and demonstrates the end-to-end water demand forecasting system for university and industrial campuses:
- **Target:** Predict hourly water consumption (liters/hour) per building.
- **Horizons:** Multi-horizon recursive rollouts for **24h, 48h, and 72h** windows.
- **Architectural Tenets:**
  1. *Shifted Computation Rule:* Zero target leakage ($y_{t-1}$ basis for all lags and rolling statistics).
  2. *Dynamic Predictions Buffer:* Closed-loop multi-step rollouts ($k > 1$) with schedule-derived future occupancy.
  3. *Physical Grounding:* Bounded strictly within $[0, 3\times \text{PeakCapacity}]$.
  4. *Tournament Rigor:* 15-model tournament, Diebold-Mariano statistical significance, and FastAPI REST endpoints.

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path for module imports
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ml.demand_forecasting.config import BUILDINGS, PATHS, TRAINING, WEATHER
from ml.demand_forecasting.data_loader import load_telemetry_data, generate_synthetic_dataset
from ml.demand_forecasting.feature_pipeline import (
    DataHygienePipeline,
    FeatureEngineer,
    IsolatedFoldPreprocessor,
    BASELINE_FEATURE_COLS,
    EXTENDED_FEATURE_COLS,
)
from ml.demand_forecasting.forecast import get_pipeline, predict
from ml.demand_forecasting.multi_horizon_engine import MultiHorizonForecaster
from ml.demand_forecasting.train import split_telemetry_data

# Styling configurations
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"font.size": 10, "figure.autolayout": True})

print(f"Working Directory: {project_root}")
print(f"Campus Buildings: {list(BUILDINGS.BUILDINGS.keys())}")

## 1. Data Ingestion & Data Hygiene

In this section, we load the continuous 6-month campus telemetry dataset (21,960 hourly observations across 5 distinct building archetypes: `academic_block_01`, `hostel_01`, `library_01`, `canteen_01`, `research_lab_01`).

We apply the `DataHygienePipeline` to enforce:
1. Hourly timestamp continuity and alignment.
2. Deduplication.
3. Linear interpolation for short sensor dropouts ($\le 3$ hours).
4. Non-imputation and flagging of long sensor dropouts ($> 3$ hours) to prevent artificial pattern distortion.

In [ ]:
# Load raw telemetry and execute data hygiene pipeline
raw_telemetry = load_telemetry_data()
print(f"Raw telemetry records: {len(raw_telemetry)} observations across {raw_telemetry['building_id'].nunique()} buildings.")

hygiene = DataHygienePipeline(short_gap_max_hours=3)
cleaned_telemetry = hygiene.process(raw_telemetry)

print(f"Cleaned telemetry shape: {cleaned_telemetry.shape}")
print("Missing water consumption values remaining:", cleaned_telemetry["water_consumption"].isna().sum())
cleaned_telemetry.head()

In [ ]:
# Visualize Diurnal Water Demand Profiles per Campus Building Archetype
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
axes = axes.flatten()

buildings = list(BUILDINGS.BUILDINGS.keys())
for i, b_id in enumerate(buildings):
    b_df = cleaned_telemetry[cleaned_telemetry["building_id"] == b_id]
    hourly_avg = b_df.groupby("hour")["water_consumption"].agg(["mean", "std"]).reset_index()
    
    ax = axes[i]
    ax.plot(hourly_avg["hour"], hourly_avg["mean"], lw=2.5, marker="o", label="Mean (L/h)")
    ax.fill_between(
        hourly_avg["hour"],
        hourly_avg["mean"] - hourly_avg["std"],
        hourly_avg["mean"] + hourly_avg["std"],
        alpha=0.2,
        label="±1 Std Dev"
    )
    b_type = BUILDINGS.BUILDINGS[b_id]["type"].replace("_", " ").title()
    ax.set_title(f"{b_id} ({b_type})", fontweight="bold")
    ax.set_xlabel("Hour of Day (0 - 23)")
    ax.set_ylabel("Consumption (L/h)")
    ax.set_xticks(range(0, 24, 4))
    ax.legend(loc="upper left", fontsize=8)

axes[-1].axis("off")  # Unused 6th subplot
plt.suptitle("Campus Diurnal Water Consumption Profiles by Archetype", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 2. Leak-Free Feature Engineering Pipeline

### Shifted Computation Rule
To guarantee **zero data leakage**, all rolling window calculations and historical lag features are computed strictly on shifted observations:
$$\text{lag\_1h}_t = y_{t-1}, \quad \text{rolling\_mean\_24h}_t = \frac{1}{24}\sum_{i=1}^{24} y_{t-i}$$
Target hour $t$ never peeks at its own ground truth.

Features computed:
- **Lags:** `lag_1h`, `lag_24h`, `lag_168h` (1 week seasonal anchor)
- **Rolling Windows:** `rolling_mean_24h`, `rolling_mean_7d`, `rolling_std_24h`, `rolling_min_24h`, `rolling_max_24h`
- **Cyclical Temporal:** Sine/Cosine transforms for hour, day of week, and month
- **Exogenous Weather:** Temperature lags (`temp_lag_1h`, `temp_lag_24h`) and cumulative precipitation (`rain_rolling_sum_24h`)

In [ ]:
# Compute extended feature matrix
fe = FeatureEngineer(use_extended_features=True)
df_features = fe.compute_features(cleaned_telemetry)

print(f"Feature matrix dimensions: {df_features.shape}")
print(f"Number of engineered features: {len(EXTENDED_FEATURE_COLS)}")
df_features[["timestamp", "building_id", "water_consumption", "lag_1h", "lag_24h", "lag_168h", "rolling_mean_24h", "hour_sin", "hour_cos"]].head()

In [ ]:
# Correlation Analysis of Engineered Features with Target Water Consumption
feature_subset = [
    "water_consumption", "lag_1h", "lag_24h", "lag_168h",
    "rolling_mean_24h", "rolling_mean_7d", "temperature",
    "rainfall", "occupancy", "is_weekend", "is_holiday"
]

corr_matrix = df_features[feature_subset].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)
plt.title("Correlation Matrix: Target Demand vs. Shifted Lags & Exogenous Features", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Chronological Train / Validation / Test Splitting

Time-series forecasting models must never use random shuffling. We enforce a **strict chronological partition per building**:
- **Train Set (70%):** Jan 1, 2026 – May 9, 2026 (15,370 rows)
- **Validation Set (15%):** May 9, 2026 – Jun 5, 2026 (3,295 rows)
- **Test Set (15%):** Jun 5, 2026 – Jul 2, 2026 (3,295 rows, held-out for final audit)

In [ ]:
# Split dataset chronologically
train_df, val_df, test_df = split_telemetry_data(df_features)

print(f"Train partition:      {len(train_df):>6} rows ({train_df['timestamp'].min()} to {train_df['timestamp'].max()})")
print(f"Validation partition: {len(val_df):>6} rows ({val_df['timestamp'].min()} to {val_df['timestamp'].max()})")
print(f"Held-Out Test split:  {len(test_df):>6} rows ({test_df['timestamp'].min()} to {test_df['timestamp'].max()})")

## 4. Multi-Model Tournament & Validation Leaderboard

We evaluated **15 candidate architectures** across baselines, regularized linear models, tree ensembles, deep learning, and stacked meta-ensembles using an expanding-window `TimeSeriesSplit` (3 folds) on the validation partition.

### Candidate Models:
- **Baselines:** Seasonal Naive ($t-168$), Moving Average (24h)
- **Linear:** Ridge, ElasticNet
- **Tree Ensembles:** Random Forest, ExtraTrees, HistGradientBoosting, LightGBM, XGBoost, CatBoost
- **Deep / Specialized:** PyTorch MLP
- **Ensembles:** Top3 Average Ensemble, Out-of-Fold Stacking Meta-Ensemble

In [ ]:
# Load validation leaderboard from tournament
with open(PATHS.OUTPUTS_DIR / "model_comparison.json", "r", encoding="utf-8") as f:
    tournament_records = json.load(f)

leaderboard_df = pd.DataFrame(tournament_records)
leaderboard_df_sorted = leaderboard_df.sort_values("mae_72h", ascending=True).reset_index(drop=True)

display_cols = [
    "model_name", "model_class", "mae_72h", "rmse_72h", "mape_72h",
    "latency_ms", "dm_p_value", "significant_vs_baseline"
]
leaderboard_df_sorted[display_cols]

In [ ]:
# Render the Leaderboard Metrics Comparison Chart
from IPython.display import Image, display

leaderboard_chart = PATHS.OUTPUTS_DIR / "leaderboard_metrics.png"
if leaderboard_chart.exists():
    display(Image(filename=str(leaderboard_chart)))

## 5. Production Pipeline & Serialized Winning Model

### Selection Rationale: CatBoost Regressor
- **Validation 72h MAE:** 70.91 L/h (55.08% improvement over baseline, $p = 0.00000$)
- **Execution Latency:** 573.8 ms (well below the < 2.0s SLA)
- **Compactness:** 339 KB total pipeline size on disk
- **Robustness:** Native handling of categorical building interactions without tree depth explosion

The production pipeline is serialized to `ml/demand_forecasting/models/winning_pipeline.joblib` alongside runner-up weights (`models/runner_ups/`) for zero-downtime model hot-swapping.

In [ ]:
# Inspect the fitted winning pipeline
pipeline = get_pipeline()

print("=" * 60)
print("PRODUCTION WINNING PIPELINE INSPECTION")
print("=" * 60)
print(f"Model Architecture:     {pipeline.get('model_name')}")
print(f"Model Version Tag:      {pipeline.get('model_version')}")
print(f"Dataset Provenance:     {pipeline.get('dataset_type')} (Hash: {pipeline.get('dataset_hash')[:12]}...)")
print(f"Training Date:          {pipeline.get('training_date')}")
print(f"Fitted Features Count:  {len(pipeline.get('transformed_feature_names', []))}")
print(f"Storage Binary Size:    {PATHS.WINNING_PIPELINE_PATH.stat().st_size / 1024.0:.1f} KB")
print(f"Registered Runner-Ups:  {list(pipeline.get('runner_up_models', {}).keys())}")

## 6. Multi-Horizon Recursive Rollout Engine Demo

The `MultiHorizonForecaster` rolls out multi-step forecasts dynamically:
1. **Step $t+1$:** Ingests `lag_1h`, `lag_24h`, and `lag_168h` directly from telemetry.
2. **Step $t+k$ ($k > 1$):** Sets `lag_1h` using its own prior prediction $\hat{y}_{t+k-1}$.
3. **Step $t+k$ ($k > 24$):** Dynamically updates `lag_24h` and `rolling_mean_24h` using intermediate rollouts.
4. **Historical Anchor:** Anchors `lag_168h` (1-week cycle) and `rolling_mean_7d` to baseline telemetry.
5. **Physical Grounding:** Strictly clamps predictions to $[0, 3 \times \text{PeakCapacity}]$.

In [ ]:
# Execute recursive rollout across 24h, 48h, and 72h horizons
engine = MultiHorizonForecaster(
    model=pipeline["model"],
    preprocessor=pipeline["preprocessor"],
    feature_engineer=pipeline["feature_engineer"],
)

b_id = "hostel_01"
hist_slice = test_df[test_df["building_id"] == b_id].iloc[:200]

print(f"Generating Multi-Horizon Rollouts for Building: {b_id}...")
for h in [24, 48, 72]:
    pred_df, source = engine.predict_horizon(
        building_id=b_id,
        historical_df=hist_slice,
        horizon_hours=h,
    )
    total_val = pred_df["predicted_consumption"].sum()
    print(f"  Horizon: {h:2d}h | Steps: {len(pred_df):2d} | Cumulative Demand: {total_val:,.1f} L | Weather: {source}")

pred_df.head(10)

## 7. Held-Out Test Set Audit & 6 Hackathon Benchmarks

We evaluate the winning CatBoost pipeline on the **locked 15% Test set** (3,295 observations).

### Target Hackathon Benchmark SLAs:
1. **Accuracy:** MAPE < 6.5% (masking actuals < 0.1 L/h).
2. **Groundedness:** 100% adherence to physical limits ($0 \le \hat{y} \le 3 \times \text{PeakCapacity}$).
3. **Latency:** < 2.0s for 72h recursive rollout.
4. **Cost:** Tracked $/m³ saved & compute overhead.
5. **False-Positive Rate:** < 5.0% surge false alarms on nominal telemetry.
6. **Task-Success Rate:** > 95.0% complete and valid forecast returns.

In [ ]:
# Load Phase E Final Audit Report (metrics.json)
metrics_json_path = PATHS.OUTPUTS_DIR / "metrics.json"
with open(metrics_json_path, "r", encoding="utf-8") as f:
    audit_report = json.load(f)

m = audit_report["primary_hackathon_metrics"]

print("=" * 70)
print("NEERARAN PHASE E AUDIT RESULTS (HELD-OUT 15% TEST SET)")
print("=" * 70)
print(f"1. Accuracy (Overall Test MAPE):  {m['1_accuracy']['overall']['mape_pct']}%  (Target: < 6.5% | Met: {m['1_accuracy']['target_met']})")
print(f"   24h MAPE: {m['1_accuracy']['horizons']['24h']['mape_pct']}% | MAE: {m['1_accuracy']['horizons']['24h']['mae']} L/h")
print(f"   48h MAPE: {m['1_accuracy']['horizons']['48h']['mape_pct']}% | MAE: {m['1_accuracy']['horizons']['48h']['mae']} L/h")
print(f"   72h MAPE: {m['1_accuracy']['horizons']['72h']['mape_pct']}% | MAE: {m['1_accuracy']['horizons']['72h']['mae']} L/h")
print(f"2. Groundedness (Adherence):     {m['2_groundedness']['compliance_pct']}% (Target: 100.0% | Met: {m['2_groundedness']['target_met']})")
print(f"3. Latency (Mean 72h Rollout):   {m['3_latency']['mean_latency_ms']} ms (Target: < 2000 ms | Met: {m['3_latency']['target_met']})")
print(f"4. Cost (Economic Utility):      ${m['4_cost']['water_saved_per_m3_usd']}/m3 saved | Storage: {m['4_cost']['model_storage_kb']} KB")
print(f"5. False-Positive Rate:          {m['5_false_positive_rate']['false_positive_rate_pct']}% (Target: < 5.0% | Met: {m['5_false_positive_rate']['target_met']})")
print(f"6. Task-Success Rate:            {m['6_task_success_rate']['task_success_rate_pct']}% (Target: > 95.0% | Met: {m['6_task_success_rate']['target_met']})")
print("=" * 70)

# Display Key Visualizations
print("\n--- Actual vs. Predicted Time-Series Overlay ---")
display(Image(filename=str(PATHS.OUTPUTS_DIR / "actual_vs_predicted.png")))

print("\n--- Horizon Error Degradation Curve (1h to 72h) ---")
display(Image(filename=str(PATHS.OUTPUTS_DIR / "horizon_error_curve.png")))

print("\n--- Feature Importance Attribution Analysis ---")
display(Image(filename=str(PATHS.OUTPUTS_DIR / "feature_importance.png")))

print("\n--- Error Residual Distribution & Q-Q Plot ---")
display(Image(filename=str(PATHS.OUTPUTS_DIR / "residual_distribution.png")))

## 8. Programmatic Python API & FastAPI Service Layer

The module exposes both an in-memory Python API (`forecast.predict`) and a REST API mounted under `/api/forecast` (`api/forecasting.py`).

In [ ]:
# 1. Programmatic Python API Invocation
result = predict(
    building_id="academic_block_01",
    horizon=72,
)
print("Programmatic API Prediction:")
print(f"  Building:          {result['building_id']} ({result['building_type']})")
print(f"  Horizon:           {result['horizon_hours']} hours")
print(f"  Total Consumption: {result['total_predicted_consumption']:,.2f} L")
print(f"  Peak Hourly Rate:  {result['peak_predicted_consumption']} L/h")
print(f"  Execution Latency: {result['execution_latency_ms']} ms")

# 2. REST API Verification via FastAPI TestClient
from fastapi.testclient import TestClient
from api.forecasting import app

client = TestClient(app)

# Health Check (Static Route Precedence)
health_res = client.get("/api/forecast/health")
print("\nREST API Health Response (GET /api/forecast/health):")
print(f"  Status: {health_res.status_code}")
print(f"  Body:   {health_res.json()}")

# Building Forecast (GET /api/forecast/{building_id})
fc_res = client.get("/api/forecast/canteen_01?horizon=24")
print("\nREST API Building Forecast (GET /api/forecast/canteen_01?horizon=24):")
print(f"  Status: {fc_res.status_code}")
print(f"  Steps:  {len(fc_res.json()['predictions'])} hourly values")

## 9. Conclusion & Operational Impact

The NEERARAN Water Demand Forecasting module achieves:
- **Accuracy:** 4.16% Test MAPE across all 5 campus archetypes (surpassing the < 6.5% goal).
- **Physical Reliability:** 100% adherence to non-negative and capacity constraints.
- **Ultra-Low Latency:** 573.8 ms mean 72h recursive rollout time (< 2.0s SLA).
- **Economic Value:** $0.15/m³ estimated water saved through autonomous pumping scheduling and overflow prevention.
- **Zero-Downtime Resilience:** Decoupled weather client with instantaneous fallback circuit breakers and runtime model switching via runner-up weights.